# Notebook 17 — Ensemble Disagreement as a Trust Map (Track 1 v1.6)

**Goal.** Convert nb16's binary "DINNDeep extrapolates poorly" verdict into a continuous **per-cell confidence map** by exploiting ensemble disagreement across random seeds.

**Hypothesis.** Per-cell standard deviation across an ensemble of `N` DINNDeep models — same architecture, same hyperparameters, same data, only `torch.manual_seed()` differs — is positively correlated with prediction error. Cells where seeds disagree are the cells the model is uncertain about. If true, this gives a per-cell trust map at inference time *without needing held-out ground truth*.

**Two experiments:**

| Section | Setup | Question |
|---|---|---|
| A | 10-seed ensemble on FULL Eq Pacific AOI | Does per-cell stdev correlate with per-cell `|error|`? |
| B | 5-seed ensemble on western 2/3 only (replicates nb16's block CV) | Is ensemble stdev higher in the held-out eastern block than in the training western blocks? |

**What outcomes mean:**

- **A passes (r > 0.4) AND B confirms (test/train stdev ratio > 2):** ensemble disagreement is a usable trust map. Production-ready: ship DINNDeep with the ensemble at inference, flag low-confidence cells.
- **A passes but B doesn't:** stdev tracks within-domain noise but not extrapolation. Trust map works for in-distribution cells only.
- **A fails:** ensemble agrees even where mean prediction is wrong. The model is overconfident; abstention via disagreement does NOT rescue extrapolation. Honest negative result that argues for physics-constrained next phase.

**Compute budget (RTX 5090).** ~7 min per DINNDeep training run. Section A = 10 seeds ≈ 75 min, Section B = 5 seeds ≈ 35 min. Total ~110 min. Designed to run overnight with intermediate checkpointing — if the kernel dies mid-run, completed seeds survive (saved as `nb17_results/seed{A,B}_NN.npz`).

**Builds on:** nb15 (DINNDeep architecture) and nb16 (block-CV verdict). The consensus / abstention idea is from Salman & Liu 2019 (arXiv:1901.06566), originally proposed for classification under cross-entropy + softmax; we adapt to regression by replacing per-class probability disagreement with per-cell prediction stdev.


In [ ]:
import sys
import time
import warnings
import json
from pathlib import Path

_repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
_src = _repo_root / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

warnings.filterwarnings("ignore", message="Couldn't find available_diagnostics.log")
warnings.filterwarnings("ignore", category=FutureWarning)

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.stats import binned_statistic_2d, pearsonr, spearmanr, mannwhitneyu

from darwindiff.carroll6 import (
    PARAM_BOUNDS,
    PARAM_NAMES,
    bounded_params,
    carroll6_step,
)
from darwindiff.diagnostics import format_pearson, safe_pearson_r
from darwindiff.ecco_darwin_loader import (
    EQUATORIAL_PACIFIC_AOI,
    open_bin_average,
    subset_aoi,
    time_mean,
)
from darwindiff.llc270_loader import (
    aoi_mask_from_xc_yc,
    list_available_iterations,
    open_llc270_tracer,
    surface_layer,
)
from darwindiff.networks import DINNDeep

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}, GPU={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no'}")

RESULTS_DIR = _repo_root / "notebooks" / "nb17_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Intermediate results: {RESULTS_DIR}")


## 1. Load FeT (target) + 4-channel covariates from bin_average

Same data and processing as nb15 / nb16 — Equatorial Pacific AOI, FeT target from the LLC270 monthly tree, SST + MLD + windSpeed + latitude as covariates from bin_average climatology.


In [ ]:
MONTHLY_ROOT = r"D:\ecco_darwin_v5\output\monthly"
GRID_DIR = r"D:\ecco_darwin_v5\grid"
BIN_AVG_PATH = r"D:\ecco_darwin_v5\bin_average\v05_ECCO-Darwin_bin_average_1x1_deg.nc"
AOI = EQUATORIAL_PACIFIC_AOI

iters = list_available_iterations(MONTHLY_ROOT, "FeT")
fet_ds = open_llc270_tracer(MONTHLY_ROOT, GRID_DIR, "FeT", iters=iters)
fet_surf = surface_layer(fet_ds)
fet_mean = fet_surf.FeT.mean(dim="time", skipna=True).values
xc_native = fet_surf.XC.values; yc_native = fet_surf.YC.values
good = aoi_mask_from_xc_yc(xc_native, yc_native, AOI.lat_min, AOI.lat_max, AOI.lon_min, AOI.lon_max) & np.isfinite(fet_mean) & (fet_mean != 0)
lat_edges = np.arange(AOI.lat_min - 0.5, AOI.lat_max + 0.5 + 0.001, 1.0)
lon_edges = np.arange(AOI.lon_min - 0.5, AOI.lon_max + 0.5 + 0.001, 1.0)
fet_binned, _, _, _ = binned_statistic_2d(yc_native[good], xc_native[good], fet_mean[good], statistic="mean", bins=[lat_edges, lon_edges])

ds_bin = open_bin_average(BIN_AVG_PATH)
eqpac_clim = time_mean(subset_aoi(ds_bin, AOI))
sst = eqpac_clim.SST.values
mld = eqpac_clim.mldDepth.values
wind = eqpac_clim.windSpeed.values
lat_1d = eqpac_clim.lat.values
lat_2d = np.broadcast_to(lat_1d[:, None], sst.shape).astype(np.float64)

ocean_mask = np.isfinite(sst) & np.isfinite(mld) & np.isfinite(wind) & np.isfinite(fet_binned)
n_ocean = int(ocean_mask.sum())
print(f"Ocean cells: {n_ocean}")

def normalize(arr):
    o = arr[ocean_mask]
    return np.where(ocean_mask, (arr - o.mean()) / max(o.std(), 1e-9), 0.0).astype(np.float32)

env_4ch = torch.tensor(np.stack([normalize(sst), normalize(mld), normalize(wind), normalize(lat_2d)], axis=0), dtype=torch.float32)
fet_target = torch.tensor(np.where(ocean_mask, fet_binned, 1.0), dtype=torch.float32)
mask_t = torch.tensor(ocean_mask, dtype=torch.bool)
H, W = env_4ch.shape[1], env_4ch.shape[2]
state0 = torch.tensor([5.0e-4, 1.0, 1.0, 0.5, 0.025]).reshape(5, 1, 1).expand(5, H, W).contiguous()

env_4ch_dev = env_4ch.to(device); state0_dev = state0.to(device)
fet_target_dev = fet_target.to(device); mask_dev = mask_t.to(device)
bounds_dev = PARAM_BOUNDS.to(device)

fet_ocean = fet_target_dev[mask_dev]
target_mean = fet_ocean.mean()
target_std = fet_ocean.std().clamp(min=1e-6)
target_z = (fet_target_dev - target_mean) / target_std
print(f"env_4ch shape: {tuple(env_4ch.shape)}, FeT z: mean={float(target_mean):.3e}, std={float(target_std):.3e}")


## Section A — Full-AOI 10-seed ensemble

Train 10 DINNDeep models on the FULL ocean mask (no hold-out). Same architecture, same hyperparameters, same data — only `torch.manual_seed()` differs across runs. Stack the predictions: shape `[N=10, H, W]`. Compute per-cell stdev = ensemble disagreement = trust map. Compare against per-cell `|error|` of the ensemble mean prediction vs Darwin truth.

**Resilience:** each seed's prediction is saved to `nb17_results/seedA_NN.npz` immediately after training. If the kernel dies mid-run, completed seeds survive — re-running this cell skips already-cached seeds.


In [ ]:
DT, N_STEPS, N_EPOCHS = 0.25, 200, 1500


def train_full_aoi(seed: int) -> dict:
    """Train DINNDeep with loss over ALL ocean cells. Mirrors nb15."""
    torch.manual_seed(seed)
    net = DINNDeep(n_input_channels=4, hidden_dim=32, n_outputs=6, n_blocks=4).to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=5e-3)
    losses = []
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(N_EPOCHS):
        optimizer.zero_grad()
        params = bounded_params(net(env_4ch_dev), bounds_dev)
        state = state0_dev
        for _ in range(N_STEPS):
            state = carroll6_step(state, params, DT)
        dfe = state[0]
        dfe_ocean = dfe[mask_dev]
        dfe_z = (dfe - dfe_ocean.mean()) / dfe_ocean.std().clamp(min=1e-6)
        residual = (dfe_z - target_z) * mask_dev.to(dfe.dtype)
        loss = (residual ** 2).sum() / mask_dev.sum().to(residual.dtype)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    if device == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - t0
    with torch.no_grad():
        params_final = bounded_params(net(env_4ch_dev), bounds_dev).cpu().numpy()
        state = state0_dev
        for _ in range(N_STEPS):
            state = carroll6_step(state, bounded_params(net(env_4ch_dev), bounds_dev), DT)
        dfe_final = state[0].cpu().numpy()
    return {
        "seed": seed,
        "losses": np.asarray(losses, dtype=np.float32),
        "params_final": params_final.astype(np.float32),
        "dfe_final": dfe_final.astype(np.float32),
        "elapsed": float(elapsed),
        "final_loss": float(losses[-1]),
    }


SEEDS_A = list(range(10))
results_A = []

for seed in SEEDS_A:
    seed_path = RESULTS_DIR / f"seedA_{seed:02d}.npz"
    if seed_path.exists():
        loaded = np.load(seed_path)
        cached = {k: loaded[k] for k in loaded.files}
        cached["seed"] = seed
        cached["final_loss"] = float(cached["final_loss"])
        cached["elapsed"] = float(cached["elapsed"])
        results_A.append(cached)
        print(f"  seedA {seed:2d}: cached at {seed_path.name}, loaded")
        continue
    print(f"  seedA {seed:2d}: training (~7 min)...", flush=True)
    res = train_full_aoi(seed)
    np.savez(
        seed_path,
        losses=res["losses"],
        params_final=res["params_final"],
        dfe_final=res["dfe_final"],
        elapsed=np.asarray(res["elapsed"], dtype=np.float32),
        final_loss=np.asarray(res["final_loss"], dtype=np.float32),
    )
    results_A.append(res)
    print(
        f"    seedA {seed:2d}: done in {res['elapsed']:.0f}s, final loss = {res['final_loss']:.4e}",
        flush=True,
    )

print(f"\n=== Section A: 10 seeds complete ===")
print(f"  Mean train time: {np.mean([r['elapsed'] for r in results_A]):.0f}s per seed")
print(f"  Final-loss mean: {np.mean([r['final_loss'] for r in results_A]):.3e}")
print(f"  Final-loss std:  {np.std([r['final_loss'] for r in results_A]):.3e}")


## 2. Compute ensemble disagreement (stdev) and correlate with prediction error


In [ ]:
preds_A = np.stack([r["dfe_final"] for r in results_A], axis=0)
pred_mean_A = preds_A.mean(axis=0)
pred_std_A = preds_A.std(axis=0)

abs_error_A = np.abs(pred_mean_A - fet_binned)

stdev_ocean = pred_std_A[ocean_mask]
abserr_ocean = abs_error_A[ocean_mask]
pr = pearsonr(stdev_ocean, abserr_ocean)
sr = spearmanr(stdev_ocean, abserr_ocean)

r_ensemble_mean = safe_pearson_r(pred_mean_A[ocean_mask], fet_binned[ocean_mask])

print("=== Section A: ensemble disagreement vs prediction error ===")
print(f"Ensemble mean r vs Darwin truth: {format_pearson(r_ensemble_mean, n_total=int(ocean_mask.sum()))}")
print()
print(f"Pearson  r(stdev, |error|): r = {pr.statistic:+.3f}  p = {pr.pvalue:.2e}")
print(f"Spearman ρ(stdev, |error|): ρ = {sr.statistic:+.3f}  p = {sr.pvalue:.2e}")

if pr.statistic > 0.6:
    verdict_A = "STRONG TRUST MAP: ensemble stdev predicts prediction error well"
elif pr.statistic > 0.3:
    verdict_A = "USABLE TRUST MAP: stdev correlates with error but with noise"
elif pr.statistic > 0.1:
    verdict_A = "WEAK TRUST MAP: stdev barely tracks error"
else:
    verdict_A = "NO TRUST MAP: ensemble disagreement does NOT track error -- model is overconfident"
print(f"\nVerdict: {verdict_A}")

summary_A = {
    "n_seeds": len(SEEDS_A),
    "ensemble_mean_r": float(r_ensemble_mean.r),
    "stdev_vs_abserr_pearson_r": float(pr.statistic),
    "stdev_vs_abserr_pearson_p": float(pr.pvalue),
    "stdev_vs_abserr_spearman_rho": float(sr.statistic),
    "stdev_vs_abserr_spearman_p": float(sr.pvalue),
    "final_losses": [float(r["final_loss"]) for r in results_A],
    "verdict": verdict_A,
}
(RESULTS_DIR / "summary_A.json").write_text(json.dumps(summary_A, indent=2))
print(f"\nWrote {RESULTS_DIR / 'summary_A.json'}")


## 3. Plots — truth, ensemble mean, ensemble stdev (trust map), |error|


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
truth_plot = np.where(ocean_mask, fet_binned, np.nan)
mean_plot = np.where(ocean_mask, pred_mean_A, np.nan)
std_plot = np.where(ocean_mask, pred_std_A, np.nan)
err_plot = np.where(ocean_mask, abs_error_A, np.nan)

im0 = axes[0, 0].imshow(truth_plot, origin="lower", aspect="auto", cmap="viridis")
axes[0, 0].set_title("Darwin FeT truth")
plt.colorbar(im0, ax=axes[0, 0])

im1 = axes[0, 1].imshow(mean_plot, origin="lower", aspect="auto", cmap="plasma")
axes[0, 1].set_title(f"Ensemble mean prediction\n(N={len(SEEDS_A)} seeds, r={r_ensemble_mean.r:.3f})")
plt.colorbar(im1, ax=axes[0, 1])

im2 = axes[1, 0].imshow(std_plot, origin="lower", aspect="auto", cmap="magma")
axes[1, 0].set_title("Ensemble stdev (TRUST MAP)\nLow=trust, high=disagreement")
plt.colorbar(im2, ax=axes[1, 0])

im3 = axes[1, 1].imshow(err_plot, origin="lower", aspect="auto", cmap="cividis")
axes[1, 1].set_title(f"|ensemble mean - truth|\n(Pearson r vs stdev: {pr.statistic:+.3f})")
plt.colorbar(im3, ax=axes[1, 1])

plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_A_trust_map_4panel.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {RESULTS_DIR / 'fig_A_trust_map_4panel.png'}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(stdev_ocean, abserr_ocean, s=8, alpha=0.4)
ax.set_xlabel("Per-cell ensemble stdev")
ax.set_ylabel("|Per-cell prediction error|")
ax.set_title(f"Trust map signal: r = {pr.statistic:+.3f}, ρ = {sr.statistic:+.3f}, n = {int(ocean_mask.sum())} cells")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_A_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {RESULTS_DIR / 'fig_A_scatter.png'}")


## Section B — Block-CV 5-seed ensemble

Same setup as nb16's block-CV (western 2/3 train, eastern 1/3 held out), but with 5 different seeds. Two questions:

1. **Reproducibility** — does block-CV failure (held-out r ≈ 0.301 in nb16) reproduce across seeds, or was nb16's verdict an unlucky single run?
2. **Trust map for extrapolation** — is per-cell stdev (computed across the 5 trained-on-W2/3 models) higher in the eastern (held-out) cells than in the western (training) cells? If yes, ensemble disagreement detects extrapolation territory.


In [ ]:
n_lon = ocean_mask.shape[1]
split_col = (n_lon * 2) // 3
lon_block_mask = np.zeros_like(ocean_mask)
lon_block_mask[:, :split_col] = True
train_mask_block = ocean_mask & lon_block_mask
test_mask_block = ocean_mask & ~lon_block_mask
print(f"Block split: train={int(train_mask_block.sum())}, test={int(test_mask_block.sum())}, split column={split_col}")


def train_with_mask(train_mask: np.ndarray, seed: int) -> dict:
    """Train DINNDeep with loss restricted to train_mask cells (per nb16)."""
    train_mask_t = torch.tensor(train_mask, dtype=torch.bool).to(device)
    fet_train = fet_target_dev[train_mask_t]
    target_mean_local = fet_train.mean()
    target_std_local = fet_train.std().clamp(min=1e-6)
    target_z_local = (fet_target_dev - target_mean_local) / target_std_local

    torch.manual_seed(seed)
    net = DINNDeep(n_input_channels=4, hidden_dim=32, n_outputs=6, n_blocks=4).to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=5e-3)
    losses = []
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(N_EPOCHS):
        optimizer.zero_grad()
        params = bounded_params(net(env_4ch_dev), bounds_dev)
        state = state0_dev
        for _ in range(N_STEPS):
            state = carroll6_step(state, params, DT)
        dfe = state[0]
        dfe_train = dfe[train_mask_t]
        dfe_z = (dfe - dfe_train.mean()) / dfe_train.std().clamp(min=1e-6)
        residual = (dfe_z - target_z_local) * train_mask_t.to(dfe.dtype)
        loss = (residual ** 2).sum() / train_mask_t.sum().to(residual.dtype)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    if device == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - t0
    with torch.no_grad():
        params_final = bounded_params(net(env_4ch_dev), bounds_dev).cpu().numpy()
        state = state0_dev
        for _ in range(N_STEPS):
            state = carroll6_step(state, bounded_params(net(env_4ch_dev), bounds_dev), DT)
        dfe_final = state[0].cpu().numpy()
    return {
        "seed": seed,
        "losses": np.asarray(losses, dtype=np.float32),
        "params_final": params_final.astype(np.float32),
        "dfe_final": dfe_final.astype(np.float32),
        "elapsed": float(elapsed),
        "final_loss": float(losses[-1]),
    }


SEEDS_B = list(range(5))
results_B = []
for seed in SEEDS_B:
    seed_path = RESULTS_DIR / f"seedB_{seed:02d}.npz"
    if seed_path.exists():
        loaded = np.load(seed_path)
        cached = {k: loaded[k] for k in loaded.files}
        cached["seed"] = seed
        cached["final_loss"] = float(cached["final_loss"])
        cached["elapsed"] = float(cached["elapsed"])
        results_B.append(cached)
        print(f"  seedB {seed:2d}: cached at {seed_path.name}, loaded")
        continue
    print(f"  seedB {seed:2d}: training (~7 min)...", flush=True)
    res = train_with_mask(train_mask_block, seed)
    np.savez(
        seed_path,
        losses=res["losses"],
        params_final=res["params_final"],
        dfe_final=res["dfe_final"],
        elapsed=np.asarray(res["elapsed"], dtype=np.float32),
        final_loss=np.asarray(res["final_loss"], dtype=np.float32),
    )
    results_B.append(res)
    print(
        f"    seedB {seed:2d}: done in {res['elapsed']:.0f}s, final loss = {res['final_loss']:.4e}",
        flush=True,
    )

print(f"\n=== Section B: 5 block-CV seeds complete ===")


## 4. Block-CV ensemble: does stdev rise in the held-out block?


In [ ]:
preds_B = np.stack([r["dfe_final"] for r in results_B], axis=0)
pred_mean_B = preds_B.mean(axis=0)
pred_std_B = preds_B.std(axis=0)

print("Per-seed train r vs held-out r (block W2/3 -> E1/3):")
print(f"{'seed':>5}  {'train r':>12}  {'held-out r':>14}")
per_seed_test_rs = []
per_seed_train_rs = []
for r in results_B:
    pred = r["dfe_final"]
    rt = safe_pearson_r(pred[train_mask_block], fet_binned[train_mask_block])
    rh = safe_pearson_r(pred[test_mask_block], fet_binned[test_mask_block])
    per_seed_train_rs.append(rt.r)
    per_seed_test_rs.append(rh.r)
    print(f"{r['seed']:>5}  {rt.r:>12.3f}  {rh.r:>14.3f}")

r_em_train_B = safe_pearson_r(pred_mean_B[train_mask_block], fet_binned[train_mask_block])
r_em_test_B = safe_pearson_r(pred_mean_B[test_mask_block], fet_binned[test_mask_block])
print(f"\nEnsemble mean: train r = {r_em_train_B.r:.3f}, held-out r = {r_em_test_B.r:.3f}")

stdev_train = pred_std_B[train_mask_block]
stdev_test = pred_std_B[test_mask_block]
print(f"\nStdev distribution:")
print(f"  Train cells (n={int(train_mask_block.sum())}):  mean={stdev_train.mean():.3e}, median={np.median(stdev_train):.3e}, p90={np.percentile(stdev_train, 90):.3e}")
print(f"  Held-out cells (n={int(test_mask_block.sum())}): mean={stdev_test.mean():.3e}, median={np.median(stdev_test):.3e}, p90={np.percentile(stdev_test, 90):.3e}")

mw = mannwhitneyu(stdev_train, stdev_test, alternative="less")
print(f"\nMann-Whitney U (one-sided, train_stdev < test_stdev): U = {mw.statistic:.3e}, p = {mw.pvalue:.2e}")

ratio = stdev_test.mean() / stdev_train.mean()
if ratio > 2.0 and mw.pvalue < 0.01:
    verdict_B = f"STRONG: held-out stdev is {ratio:.1f}× the train stdev -- ensemble disagreement detects extrapolation territory"
elif ratio > 1.2:
    verdict_B = f"WEAK: held-out stdev is {ratio:.2f}× train stdev (not a strong separator)"
else:
    verdict_B = f"FAILS: held-out stdev ({stdev_test.mean():.3e}) <= train stdev ({stdev_train.mean():.3e}) -- ensemble is OVERCONFIDENT in extrapolation territory"
print(f"\nVerdict: {verdict_B}")

summary_B = {
    "n_seeds": len(SEEDS_B),
    "ensemble_train_r": float(r_em_train_B.r),
    "ensemble_test_r": float(r_em_test_B.r),
    "per_seed_train_r": [float(x) for x in per_seed_train_rs],
    "per_seed_test_r": [float(x) for x in per_seed_test_rs],
    "stdev_train_mean": float(stdev_train.mean()),
    "stdev_test_mean": float(stdev_test.mean()),
    "stdev_ratio_test_train": float(ratio),
    "mann_whitney_p_value": float(mw.pvalue),
    "verdict": verdict_B,
}
(RESULTS_DIR / "summary_B.json").write_text(json.dumps(summary_B, indent=2))
print(f"\nWrote {RESULTS_DIR / 'summary_B.json'}")


## 5. Plots — block-CV ensemble: predictions, stdev, train-vs-held-out distributions, per-seed reproducibility


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

mean_plot_B = np.where(ocean_mask, pred_mean_B, np.nan)
im0 = axes[0, 0].imshow(mean_plot_B, origin="lower", aspect="auto", cmap="plasma")
axes[0, 0].axvline(split_col - 0.5, color="white", linestyle="--", lw=2, label="W2/3 vs E1/3 split")
axes[0, 0].set_title(f"Block-CV ensemble mean (N=5)\ntrain r={r_em_train_B.r:.3f}, held-out r={r_em_test_B.r:.3f}")
axes[0, 0].legend(loc="upper right", fontsize=9)
plt.colorbar(im0, ax=axes[0, 0])

std_plot_B = np.where(ocean_mask, pred_std_B, np.nan)
im1 = axes[0, 1].imshow(std_plot_B, origin="lower", aspect="auto", cmap="magma")
axes[0, 1].axvline(split_col - 0.5, color="white", linestyle="--", lw=2)
axes[0, 1].set_title(f"Block-CV ensemble stdev\nratio test/train = {ratio:.2f}")
plt.colorbar(im1, ax=axes[0, 1])

bins = np.linspace(0, np.percentile(np.concatenate([stdev_train, stdev_test]), 99), 40)
axes[1, 0].hist(stdev_train, bins=bins, alpha=0.6, label=f"Train cells (n={len(stdev_train)})", color="tab:blue", density=True)
axes[1, 0].hist(stdev_test, bins=bins, alpha=0.6, label=f"Held-out cells (n={len(stdev_test)})", color="tab:red", density=True)
axes[1, 0].set_xlabel("Per-cell stdev")
axes[1, 0].set_ylabel("Density")
axes[1, 0].set_title(f"Stdev distribution (Mann-Whitney p = {mw.pvalue:.2e})")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

x = np.arange(len(SEEDS_B))
w = 0.35
axes[1, 1].bar(x - w/2, per_seed_train_rs, w, label="Train r", color="tab:green")
axes[1, 1].bar(x + w/2, per_seed_test_rs, w, label="Held-out r", color="tab:orange")
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels([f"seed {s}" for s in SEEDS_B])
axes[1, 1].set_ylim(0, 1.05)
axes[1, 1].set_ylabel("Pearson r")
axes[1, 1].axhline(0.301, color="gray", linestyle=":", label="nb16 single-run held-out (0.301)")
axes[1, 1].set_title("Per-seed reproducibility of nb16 result")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_B_block_cv_4panel.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {RESULTS_DIR / 'fig_B_block_cv_4panel.png'}")


## Interpretation and what this means for the next phase

The two sub-experiments answer different but complementary questions:

| Section | Question | Outcome → Implication |
|---|---|---|
| A | Does ensemble stdev (full-AOI training) predict per-cell error? | If yes (r > 0.4): ship DINNDeep with the ensemble at inference; per-cell stdev becomes a usable trust score. If no: the ensemble is overconfident even on its training distribution. |
| B | Does ensemble stdev rise on held-out (extrapolation) cells? | If yes (test/train ratio > 2): ensemble disagreement detects extrapolation territory — flags cross-basin transfer cells without ground truth. If no: trust map only works in-distribution; extrapolation flagging needs physics constraints (or a different approach). |

**For the email to Jonathan.** The headline becomes "DINNDeep extrapolation failure (nb16) is/isn't detectable at inference time without held-out ground truth via ensemble disagreement." Either outcome is a real result worth sharing.

**Connections:**
- Builds on **nb15** (DINNDeep architecture) and **nb16** (block-CV verdict).
- The consensus / abstention idea is from Salman & Liu 2019 ([arXiv:1901.06566](https://arxiv.org/abs/1901.06566)). They proposed it for classification under cross-entropy + softmax; we adapt to regression by replacing per-class probability disagreement with per-cell prediction stdev.
- All artifacts (per-seed predictions, summary JSONs, figures) are persisted in `notebooks/nb17_results/` so this can be re-analyzed without retraining.

**Where this fits in the project arc:**
- 09–14: SST-only DINN fits across multiple (AOI × target) combos
- 15: architecture upgrade test (Track 1 v1.4)
- 16: cross-validation honesty check on nb15 (Track 1 v1.5)
- **17 (this notebook): ensemble disagreement as inference-time trust map** (Track 1 v1.6)
- After this: cluster transfer, box-model carbonate extension, Track 2.
